In [1]:
# ─── CELL 1: Install / upgrade dependencies ────────────────

# !pip install -q transformers==4.40.0 datasets accelerate evaluate scikit-learn tqdm

In [2]:
# ─── CELL 2: Imports ───────────────────────────────────────

import os, gc, math, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    average_precision_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

In [3]:
# ─── CELL 3: Reproducibility & device setup ────────────────

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

NUM_GPUS = torch.cuda.device_count()
print(f"GPUs available: {NUM_GPUS}")
for i in range(NUM_GPUS):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Primary device: {DEVICE}")

GPUs available: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4
Primary device: cuda


In [4]:
# ─── CELL 4: Configuration ─────────────────────────────────

MODEL_NAME  = "jjzha/jobbert-base-cased"
OUTPUT_DIR  = "/kaggle/working/jobbert_output"
LOGGING_DIR = "/kaggle/working/jobbert_logs"

# Paths
TRAIN_CSV = (
    "/kaggle/input/datasets/danielantoniudumitru/"
    "talentclef-subtaska/Training/normalization/"
    "normalized_job_applicant_dataset.csv"
)
DEV_BASE   = "/kaggle/input/datasets/danielantoniudumitru/talentclef-subtaska/Development/en"
CORPUS_DIR = os.path.join(DEV_BASE, "corpus")
QUERY_DIR  = os.path.join(DEV_BASE, "queries")
QRELS_FILE = os.path.join(DEV_BASE, "qrels.tsv")

# Hyper-parameters
MAX_LEN        = 512
TRAIN_BATCH    = 8
EVAL_BATCH     = 16
GRAD_ACCUM     = 4
NUM_EPOCHS     = 4
LR             = 2e-5
WARMUP_RATIO   = 0.1
WEIGHT_DECAY   = 0.01

os.makedirs(OUTPUT_DIR,  exist_ok=True)
os.makedirs(LOGGING_DIR, exist_ok=True)

In [5]:
# ─── CELL 5: Load & inspect training data ──────────────────

print("Loading training CSV …")
df_train = pd.read_csv(TRAIN_CSV)
print(f"Shape : {df_train.shape}")
print(df_train.head(3))
print("\nColumn dtypes:\n", df_train.dtypes)
print("\nValue counts (label):\n", df_train.iloc[:, -1].value_counts())

Loading training CSV …
Shape : (10000, 3)
                                              Resume  \
0  Proficient in Injury Prevention, Motivation, N...   
1  Proficient in Healthcare, Pharmacology, Medica...   
2  Proficient in Forecasting, Financial Modelling...   

                                     Job Description  Best Match  
0  Fitness Coach\n\n A Fitness Coach is responsib...           0  
1  Physician\n\nDiagnose and treat illnesses, pre...           0  
2  Financial Analyst\n\nAs a Financial Analyst, y...           0  

Column dtypes:
 Resume             object
Job Description    object
Best Match          int64
dtype: object

Value counts (label):
 Best Match
0    5150
1    4850
Name: count, dtype: int64


In [6]:
# ─── CELL 6: Define columns (hardcoded for this dataset) ───

cols = df_train.columns.tolist()

text_a_col = "Job Description"
text_b_col = "Resume"
label_col  = "Best Match"

print(f"Text A column : {text_a_col}")
print(f"Text B column : {text_b_col}")
print(f"Label column  : {label_col}")

# Ensure binary integer labels 0/1
df_train[label_col] = df_train[label_col].astype(int)
assert set(df_train[label_col].unique()).issubset({0, 1}), \
    "Label column must contain only 0 and 1!"

Text A column : Job Description
Text B column : Resume
Label column  : Best Match


In [7]:
# ─── CELL 7: Train / validation split ──────────────────────

df_train_split, df_val_split = train_test_split(
    df_train, test_size=0.1, stratify=df_train[label_col], random_state=SEED
)
print(f"Train size : {len(df_train_split)}")
print(f"Val size : {len(df_val_split)}")

Train size : 9000
Val size : 1000


In [8]:
# ─── CELL 8: Tokenizer ─────────────────────────────────────

print(f"\nLoading tokenizer: {MODEL_NAME} …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


Loading tokenizer: jjzha/jobbert-base-cased …


config.json:   0%|          | 0.00/603 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [9]:
# ─── CELL 9: Dataset class ─────────────────────────────────

class PairDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len,
                 col_a, col_b, label_col):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.col_a = col_a
        self.col_b = col_b
        self.label_col = label_col

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row  = self.data.iloc[idx]

        def clean_text(t):
            return " ".join(str(t).split()[:256]) if pd.notna(t) else ""

        text_a = clean_text(row[self.col_a]) if pd.notna(row[self.col_a]) else ""
        text_b = clean_text(row[self.col_b]) if pd.notna(row[self.col_b]) else ""
        
        encoding = self.tokenizer(
            text_a,
            text_b,
            max_length=self.max_len,
            truncation=True,
            padding=False,
            return_token_type_ids=True,
        )
        encoding["labels"] = int(row[self.label_col])
        return encoding


train_dataset = PairDataset(df_train_split, tokenizer, MAX_LEN,
                             text_a_col, text_b_col, label_col)
val_dataset   = PairDataset(df_val_split,  tokenizer, MAX_LEN,
                             text_a_col, text_b_col, label_col)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"Sample encoding keys: {list(train_dataset[0].keys())}")

Sample encoding keys: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']


In [10]:
# ─── CELL 10: Model ────────────────────────────────────────

print(f"\nLoading model: {MODEL_NAME} …")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")


Loading model: jjzha/jobbert-base-cased …


model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: jjzha/jobbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider train

Model parameters: 108,311,810


In [11]:
# ─── CELL 11: Metrics ──────────────────────────────────────

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs  = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    preds  = (probs >= 0.5).astype(int)

    return {
        "accuracy"  : accuracy_score(labels, preds),
        "f1"        : f1_score(labels, preds, zero_division=0),
        "precision" : precision_score(labels, preds, zero_division=0),
        "recall"    : recall_score(labels, preds, zero_division=0),
        "ap"        : average_precision_score(labels, probs),
    }

In [12]:
# ─── CELL 12: TrainingArguments ────────────────────────────

steps_per_epoch = math.ceil(len(train_dataset) / (TRAIN_BATCH * max(NUM_GPUS, 1) * GRAD_ACCUM))
total_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

os.environ["TENSORBOARD_LOGGING_DIR"] = LOGGING_DIR

training_args = TrainingArguments(
    output_dir = OUTPUT_DIR,
    logging_dir = LOGGING_DIR,

    # Epochs & batch
    num_train_epochs = NUM_EPOCHS,
    per_device_train_batch_size = TRAIN_BATCH,
    per_device_eval_batch_size = EVAL_BATCH,
    gradient_accumulation_steps = GRAD_ACCUM,

    # Optimiser
    learning_rate = LR,
    weight_decay = WEIGHT_DECAY,
    warmup_steps = warmup_steps,
    lr_scheduler_type = "cosine",

    # Evaluation & saving
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    metric_for_best_model = "ap",
    greater_is_better = True,

    # Logging
    report_to = "none",

    # Multi-GPU / precision
    fp16 = True,
    dataloader_num_workers = 4,
    seed = SEED,

    # Progress bar
    disable_tqdm = False
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [13]:
# ─── CELL 13: Trainer ──────────────────────────────────────

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    processing_class = tokenizer,
    data_collator = data_collator,
    compute_metrics = compute_metrics,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=2)]
)

In [14]:
# ─── CELL 14: Training ─────────────────────────────────────

print("\n" + "="*60)
print("  Starting JobBERT fine-tuning …")
print("="*60)

train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("\nTraining summary:")
print(train_result.metrics)


  Starting JobBERT fine-tuning …


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Ap
1,No log,1.370506,0.506000,0.587646,0.493689,0.725773,0.493698
2,No log,1.367855,0.512000,0.271642,0.491892,0.187629,0.498138
3,No log,1.375110,0.515000,0.211382,0.500000,0.134021,0.505130
4,5.460890,1.368915,0.500000,0.496982,0.485265,0.509278,0.505110


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training summary:
{'train_runtime': 909.6172, 'train_samples_per_second': 39.577, 'train_steps_per_second': 0.62, 'total_flos': 3764436757901760.0, 'train_loss': 5.44594406236148, 'epoch': 4.0}


In [15]:
# ─── CELL 15: Helper — read dev corpus / queries ───────────

def read_text_file(folder, file_id):
    path = os.path.join(folder, str(file_id))
    if not os.path.exists(path):
        return ""
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return f.read().strip()

In [16]:
# ─── CELL 16: Load qrels ───────────────────────────────────

print("\nLoading qrels …")
qrels = pd.read_csv(
    QRELS_FILE, sep="\t", header=None,
    names=["q_id", "iter", "c_id", "relevance"]
)
print(f"qrels shape: {qrels.shape}")
print(qrels.head())


Loading qrels …
qrels shape: (472, 4)
    q_id  iter   c_id  relevance
0  36044     0  13884          1
1  39060     0   9516          1
2  39060     0  12097          1
3  32447     0  13882          1
4  39060     0   6533          1


In [17]:
# ─── CELL 17: Build dev pairs ──────────────────────────────

print("\nBuilding dev (query, corpus) pairs …")
dev_records = []

for _, row in tqdm(qrels.iterrows(), total=len(qrels), desc="Reading files"):
    q_text = read_text_file(QUERY_DIR,  row["q_id"])
    c_text = read_text_file(CORPUS_DIR, row["c_id"])
    dev_records.append({
        "q_id" : row["q_id"],
        "c_id" : row["c_id"],
        "query_text": q_text,
        "corpus_text": c_text,
        "relevance" : int(row["relevance"]),
    })

all_q_ids = qrels["q_id"].unique().tolist()
all_c_ids = qrels["c_id"].unique().tolist()
positive_pairs = set(zip(qrels["q_id"], qrels["c_id"]))

negatives = []
for q_id in all_q_ids:
    neg_c_ids = [c for c in all_c_ids if (q_id, c) not in positive_pairs]
    sampled = random.sample(neg_c_ids, min(len(neg_c_ids), 10))
    for c_id in sampled:
        q_text = read_text_file(QUERY_DIR, q_id)
        c_text = read_text_file(CORPUS_DIR, c_id)
        negatives.append({
            "q_id": q_id, "c_id": c_id,
            "query_text": q_text, "corpus_text": c_text,
            "relevance": 0,
        })

df_dev = pd.DataFrame(dev_records + negatives)
print(f"Dev pairs: {len(df_dev)}")
print(df_dev.head(2))


Building dev (query, corpus) pairs …


Reading files:   0%|          | 0/472 [00:00<?, ?it/s]

Dev pairs: 572
    q_id   c_id                                         query_text  \
0  36044  13884  Failure Analysis Engineer\n\nIntroduction\nWe ...   
1  39060   9516  Sales Director, ProductFE\n\nWe are seeking a ...   

                                         corpus_text  relevance  
0  Siti Wijaya\n\nJalan Merdeka 42, Bandung 40173...          1  
1  Fatima Ahmed\nHassan\n+249-912-345-678\nfatima...          1  


In [18]:
# ─── CELL 18: Dev Dataset class ────────────────────────────

class DevPairDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        def clean_text(t):
            return " ".join(str(t).split()[:256]) if pd.notna(t) else ""

        row = self.data.iloc[idx]
        enc = self.tokenizer(
            clean_text(row["query_text"]),
            clean_text(row["corpus_text"]),
            max_length=self.max_len,
            truncation=True,
            padding=False,
            return_token_type_ids=True,
        )
        enc["labels"] = int(row["relevance"])
        return enc

dev_dataset = DevPairDataset(df_dev, tokenizer, MAX_LEN)

In [19]:
# ─── CELL 19: Inference on dev set ─────────────────────────

print("\nRunning predictions on dev set …")

raw_preds = trainer.predict(dev_dataset)
logits = raw_preds.predictions
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
preds_bin = (probs >= 0.5).astype(int)
labels = raw_preds.label_ids

df_dev["pred_prob"]  = probs
df_dev["pred_label"] = preds_bin


Running predictions on dev set …


In [20]:
# ─── CELL 20: Ranking Metrics ──────────────────────────────

def mean_average_precision(df, q_col="q_id", prob_col="pred_prob", rel_col="relevance"):
    aps = []
    for qid, grp in df.groupby(q_col):
        grp_sorted = grp.sort_values(prob_col, ascending=False).reset_index(drop=True)
        rel = grp_sorted[rel_col].values
        if rel.sum() == 0:
            continue
        hits, ap = 0, 0.0
        for rank, r in enumerate(rel, start=1):
            if r == 1:
                hits += 1
                ap += hits / rank
        aps.append(ap / rel.sum())
    return np.mean(aps) if aps else 0.0


def mean_reciprocal_rank(df, q_col="q_id", prob_col="pred_prob", rel_col="relevance"):
    rrs = []
    for qid, grp in df.groupby(q_col):
        grp_sorted = grp.sort_values(prob_col, ascending=False).reset_index(drop=True)
        rel = grp_sorted[rel_col].values
        rr = 0.0
        for rank, r in enumerate(rel, start=1):
            if r == 1:
                rr = 1.0 / rank
                break
        rrs.append(rr)
    return np.mean(rrs) if rrs else 0.0


def precision_at_k(df, k, q_col="q_id", prob_col="pred_prob", rel_col="relevance"):
    pk_list = []
    for qid, grp in df.groupby(q_col):
        grp_sorted = grp.sort_values(prob_col, ascending=False).head(k)
        pk_list.append(grp_sorted[rel_col].mean())
    return np.mean(pk_list) if pk_list else 0.0

In [21]:
# ─── CELL 21: Print all metrics ────────────────────────────

acc  = accuracy_score(labels, preds_bin)
f1   = f1_score(labels, preds_bin, zero_division=0)
prec = precision_score(labels, preds_bin, zero_division=0)
rec  = recall_score(labels, preds_bin, zero_division=0)
ap_overall = average_precision_score(labels, probs)

MAP   = mean_average_precision(df_dev)
MRR   = mean_reciprocal_rank(df_dev)
P_at1 = precision_at_k(df_dev, k=1)
P_at5 = precision_at_k(df_dev, k=5)
P_at10= precision_at_k(df_dev, k=10)

print("\n" + "="*60)
print("  JobBERT — Dev Set Evaluation Results")
print("="*60)
print(f"  Accuracy: {acc:.4f}")
print(f"  F1: {f1:.4f}")
print(f"  Precision (binary): {prec:.4f}")
print(f"  Recall: {rec:.4f}")
print(f"  Average Precision: {ap_overall:.4f}")
print("─"*60)
print(f"  MAP(Mean Avg Precision): {MAP:.4f}")
print(f"  MRR(Mean Recip Rank): {MRR:.4f}")
print(f"  P@1: {P_at1:.4f}")
print(f"  P@5: {P_at5:.4f}")
print(f"  P@10: {P_at10:.4f}")
print("="*60)


  JobBERT — Dev Set Evaluation Results
  Accuracy: 0.1748
  F1: 0.0000
  Precision (binary): 0.0000
  Recall: 0.0000
  Average Precision: 0.8824
────────────────────────────────────────────────────────────
  MAP(Mean Avg Precision): 0.7449
  MRR(Mean Recip Rank): 0.5517
  P@1: 0.4000
  P@5: 0.5400
  P@10: 0.6600


In [22]:
# ─── CELL 22: Save predictions ─────────────────────────────

df_dev.to_csv("/kaggle/working/jobbert_dev_predictions.csv", index=False)
print("\nPredictions saved to /kaggle/working/jobbert_dev_predictions.csv")


Predictions saved to /kaggle/working/jobbert_dev_predictions.csv
